# Experiment 1C — Pretrained Model Experiments

Three sub-experiments exploring pretrained models as baselines and hybrid architectures:

| Sub-Exp | Description | Architecture |
|---|---|---|
| **1C-i** | Single pretrained models (fine-tuned) | EfficientNet-B0, MobileNetV3-Large |
| **1C-ii** | Paper's MHA head + pretrained conv blocks | EfficientNet features → project → MHA×2 → MLP |
| **1C-iii** | MHA variants (placeholder) | TODO — custom MHA configs |

**All use:**
- Sequential per-class split (no leakage)
- Grayscale input (weight-averaged from ImageNet RGB weights)
- AdamW optimizer, cosine LR, warmup

**Purpose:** Establish pretrained baselines and test whether the paper's attention head
benefits from better feature extraction (pretrained conv blocks vs custom CNNs).

---
## 0 · Configuration & Imports

In [ ]:
import os, sys, json, random, csv, re, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tv_models
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from PIL import Image

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = "/home/metal/Experiment/data/The IQ-OTHNCCD lung cancer dataset"
SPLITS_CSV   = Path("../../00_dataset/splits.csv")
RESULTS_DIR  = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────────
CLASS_FOLDERS  = ["Benign cases", "Malignant cases", "Normal cases"]
CLASS_TO_LABEL = {"Benign cases": 0, "Malignant cases": 1, "Normal cases": 2}
LABEL_TO_CLASS = {0: "Benign", 1: "Malignant", 2: "Normal"}
NUM_CLASSES    = 3
IMG_SIZE       = 224
TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15
BATCH_SIZE     = 32
NUM_WORKERS    = 4
EPOCHS         = 30         # Pretrained models converge faster
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
GRAD_CLIP      = 1.0
WARMUP_EPOCHS  = 5
EARLY_STOP_PAT = 10

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

---
## 1 · Dataset & DataLoader (shared across all 1C sub-experiments)

In [ ]:
def extract_case_id(f):
    m = re.search(r'\((\d+)\)', f)
    return int(m.group(1)) if m else abs(hash(f)) % 100_000

def load_splits_csv(path=SPLITS_CSV):
    with open(path) as f:
        rows = list(csv.DictReader(f))
    for r in rows: r['label'] = int(r['label']); r['case_id'] = int(r['case_id'])
    return rows

def build_case_registry(root=DATASET_ROOT):
    root = Path(root); records = []
    for cls_folder in CLASS_FOLDERS:
        d = root / cls_folder
        if not d.exists(): continue
        lbl = CLASS_TO_LABEL[cls_folder]
        for f in sorted(d.iterdir()):
            if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                records.append({'image_path': str(f), 'class_name': cls_folder,
                                'label': lbl, 'case_id': extract_case_id(f.name),
                                'filename': f.name})
    return records

def sequential_split(records):
    by_cls = defaultdict(list)
    for r in records: by_cls[r['class_name']].append(r)
    out = []
    for cls, recs in by_cls.items():
        recs = sorted(recs, key=lambda r: r['case_id'])
        n = len(recs); nt = int(n*TRAIN_RATIO); nv = int(n*VAL_RATIO)
        for i, r in enumerate(recs):
            r2 = dict(r); r2['split'] = 'train' if i<nt else ('val' if i<nt+nv else 'test')
            out.append(r2)
    return out

def get_class_weights(records):
    train = [r for r in records if r['split'] == 'train']
    cnt = Counter(r['label'] for r in train); total = sum(cnt.values())
    w = torch.zeros(NUM_CLASSES)
    for lbl, c in cnt.items(): w[lbl] = total / (NUM_CLASSES * c)
    return w

def verify_no_leakage(records):
    case_splits = defaultdict(set)
    for r in records: case_splits[r['case_id']].add(r['split'])
    leaks = {c: s for c, s in case_splits.items() if len(s) > 1}
    if leaks:
        print(f"❌ Patient leakage detected! {len(leaks)} cases")
        return False
    print(f"✅ No leakage — {len(case_splits)} cases each in one split.")
    return True

def get_transforms(split, augment=False):
    base = [T.Resize((IMG_SIZE, IMG_SIZE)), T.Grayscale(1)]
    aug  = ([T.RandomHorizontalFlip(0.5), T.RandomRotation(15),
             T.RandomAffine(0, translate=(0.05, 0.05)), T.ColorJitter(0.15, 0.15)]
            if split == 'train' and augment else [])
    return T.Compose(base + aug + [T.ToTensor(), T.Normalize([0.5], [0.5])])

class CTDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records; self.transform = transform
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        return self.transform(Image.open(r['image_path']).convert('RGB')), r['label']

def make_loaders(records, batch_size=BATCH_SIZE, augment_train=False):
    loaders = {}
    for sp in ['train', 'val', 'test']:
        sub = [r for r in records if r['split'] == sp]
        tf  = get_transforms(sp, augment=(sp == 'train' and augment_train))
        loaders[sp] = DataLoader(CTDataset(sub, tf), batch_size=batch_size,
                                  shuffle=(sp == 'train'), num_workers=NUM_WORKERS,
                                  pin_memory=True)
    return loaders['train'], loaders['val'], loaders['test']


# ── Load data ────────────────────────────────────────────────────────────────
if SPLITS_CSV.exists():
    print(f"Loading splits.csv from {SPLITS_CSV.resolve()}")
    split_records = load_splits_csv()
else:
    print("⚠️  splits.csv not found — computing sequential split...")
    split_records = sequential_split(build_case_registry())

verify_no_leakage(split_records)

for sp in ['train', 'val', 'test']:
    sub = [r for r in split_records if r['split'] == sp]
    print(f"  {sp:<6}: {len(sub)} imgs | {dict(Counter(r['class_name'] for r in sub))}")

class_weights = get_class_weights(split_records)
print(f"Class weights: {class_weights.tolist()}")

---
## 2 · Training Engine (shared)

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, SequentialLR, LinearLR

def train_one_epoch(model, loader, opt, crit, device, grad_clip):
    model.train(); ls = corr = tot = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad(); logits = model(imgs); loss = crit(logits, labels)
        loss.backward()
        if grad_clip: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        opt.step()
        ls += loss.item()*imgs.size(0); corr += (logits.argmax(1)==labels).sum().item(); tot += imgs.size(0)
    return ls/tot, corr/tot

@torch.no_grad()
def evaluate(model, loader, crit, device):
    model.eval(); ls = corr = tot = 0; preds=[]; labels_=[]; probs=[]
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs); loss = crit(logits, labels)
        ls += loss.item()*imgs.size(0); corr += (logits.argmax(1)==labels).sum().item(); tot += imgs.size(0)
        preds.extend(logits.argmax(1).cpu().tolist())
        labels_.extend(labels.cpu().tolist())
        probs.extend(F.softmax(logits, 1).cpu().tolist())
    return ls/tot, corr/tot, preds, labels_, probs

def compute_metrics(preds, labels, probs):
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    rep = classification_report(labels, preds, target_names=names, output_dict=True, zero_division=0)
    try: auc_val = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except: auc_val = float('nan')
    return {'accuracy': rep['accuracy'], 'f1_macro': rep['macro avg']['f1-score'],
            'auc_macro': auc_val, 'per_class': {n: rep[n] for n in names}}

def plot_curves(tl, vl, ta, va, path):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(tl, 'b', label='Train'); ax[0].plot(vl, 'r', label='Val')
    ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(True)
    ax[1].plot(ta, 'b', label='Train'); ax[1].plot(va, 'r', label='Val')
    ax[1].set_title('Accuracy'); ax[1].legend(); ax[1].grid(True)
    plt.tight_layout(); plt.savefig(path, dpi=100, bbox_inches='tight'); plt.show()

def plot_confusion(preds, labels, path, title=''):
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]; cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(names)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.colorbar(im); plt.tight_layout(); plt.savefig(path, dpi=100, bbox_inches='tight'); plt.show()

def plot_roc(labels, probs, path, title='ROC'):
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    y_bin = label_binarize(labels, classes=list(range(NUM_CLASSES)))
    probs_arr = np.array(probs)
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, name in enumerate(names):
        if y_bin[:, i].sum() == 0: continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs_arr[:, i])
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    ax.plot([0,1],[0,1],'k--', alpha=0.3)
    ax.set(title=title, xlabel='FPR', ylabel='TPR')
    ax.legend(); plt.tight_layout(); plt.savefig(path, dpi=100, bbox_inches='tight'); plt.show()

def run_experiment(model, train_loader, val_loader, test_loader,
                   save_dir, model_name, class_weights,
                   epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                   grad_clip=GRAD_CLIP, warmup_epochs=WARMUP_EPOCHS,
                   early_stop_pat=EARLY_STOP_PAT, device=DEVICE, smoke_test=False):
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    if smoke_test: epochs = 2
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n🏗  {model_name} | {n_params/1e6:.2f}M params | {epochs} epochs")

    crit  = nn.CrossEntropyLoss(weight=class_weights.to(device))
    opt   = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    warmup = LinearLR(opt, start_factor=0.1, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(opt, T_max=max(1, epochs - warmup_epochs))
    sched  = SequentialLR(opt, [warmup, cosine], milestones=[warmup_epochs])

    best_f1 = patience = 0
    tl, vl, ta, va = [], [], [], []
    for epoch in range(epochs):
        t0 = time.time()
        tr_l, tr_a = train_one_epoch(model, train_loader, opt, crit, device, grad_clip)
        vl_l, vl_a, vp, vlab, vprob = evaluate(model, val_loader, crit, device)
        vm = compute_metrics(vp, vlab, vprob); sched.step()
        tl.append(tr_l); vl.append(vl_l); ta.append(tr_a); va.append(vl_a)
        marker = ''
        if vm['f1_macro'] > best_f1:
            best_f1 = vm['f1_macro']; patience = 0
            torch.save(model.state_dict(), save_dir / f"{model_name}_best.pth")
            marker = ' ★'
        else:
            patience += 1
            if patience >= early_stop_pat:
                print(f"  ⏹  Early stop @ {epoch+1}"); break
        print(f"  [{epoch+1:3d}/{epochs}] tr={tr_l:.4f}/{tr_a:.4f} "
              f"vl={vl_l:.4f}/{vl_a:.4f} f1={vm['f1_macro']:.4f} ({time.time()-t0:.1f}s){marker}")

    ckpt = save_dir / f"{model_name}_best.pth"
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=device))
    _, _, tp, tlab, tprob = evaluate(model, test_loader, crit, device)
    test_m = compute_metrics(tp, tlab, tprob)
    test_m['params_M'] = round(n_params / 1e6, 3)

    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    print("\n" + "=" * 60)
    print(f"  TEST — {model_name}")
    print("=" * 60)
    print(classification_report(tlab, tp, target_names=names, zero_division=0))

    plot_curves(tl, vl, ta, va, save_dir / f"{model_name}_curves.png")
    plot_confusion(tp, tlab, save_dir / f"{model_name}_confusion.png", title=model_name)
    plot_roc(tlab, tprob, save_dir / f"{model_name}_roc.png", title=f"{model_name} — ROC")
    with open(save_dir / 'metrics.json', 'w') as f: json.dump(test_m, f, indent=2)
    return test_m

print("✅ Training engine defined.")

---
## 3 · Model Builders

### 3a. Single Pretrained Models (1C-i)
First conv adapted from 3→1 channel by averaging ImageNet RGB weights.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1C-i: Single Pretrained Models
# ═══════════════════════════════════════════════════════════════════════════════

def build_efficientnet_b0(num_classes=NUM_CLASSES):
    """EfficientNet-B0 with 3→1 channel adaptation."""
    model = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    old = model.features[0][0]
    new = nn.Conv2d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
    with torch.no_grad():
        new.weight = nn.Parameter(old.weight.mean(dim=1, keepdim=True))
    model.features[0][0] = new
    in_f = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.2, inplace=True), nn.Linear(in_f, num_classes))
    n = sum(p.numel() for p in model.parameters())
    print(f"✅ EfficientNet-B0    | {n/1e6:.2f}M params")
    return model


def build_mobilenet_v3(num_classes=NUM_CLASSES):
    """MobileNetV3-Large with 3→1 channel adaptation."""
    model = tv_models.mobilenet_v3_large(weights=tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    old = model.features[0][0]
    new = nn.Conv2d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
    with torch.no_grad():
        new.weight = nn.Parameter(old.weight.mean(dim=1, keepdim=True))
    model.features[0][0] = new
    in_f = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_f, num_classes)
    n = sum(p.numel() for p in model.parameters())
    print(f"✅ MobileNetV3-Large  | {n/1e6:.2f}M params")
    return model

### 3b. Hybrid: Pretrained Conv Blocks + Paper's MHA Head (1C-ii)

Replace the paper's custom CNN_Model1/CNN_Model2 with pretrained feature extractors,
but keep the same MHA×2 + MLP head from the paper.

**Architecture:**
```
Input (1×224×224)
    ├─ EfficientNet-B0 features → AdaptiveAvgPool → (B, 1280) ──► deep branch
    └─ MobileNetV3 early features → AdaptiveAvgPool → (B, 40)  ──► shallow branch
                         concat → (B, 1320)
                         Linear → (B, T×E)  reshape → (B, T, E)
                         MHA × 2 → GAP → BN → Dense(256) → GELU → BN → Dense(3)
```

This tests: **Does the paper's attention head benefit from better features?**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1C-ii: Hybrid Model — Pretrained Conv Blocks + Paper's MHA Head
# ═══════════════════════════════════════════════════════════════════════════════

class PretrainedDeepBranch(nn.Module):
    """EfficientNet-B0 feature extractor (replaces CNN_Model1).
    Output: (B, 1280) global features."""
    def __init__(self, freeze_backbone=False):
        super().__init__()
        effnet = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        # Adapt first conv: 3→1 channel
        old = effnet.features[0][0]
        new = nn.Conv2d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        with torch.no_grad():
            new.weight = nn.Parameter(old.weight.mean(dim=1, keepdim=True))
        effnet.features[0][0] = new
        self.features = effnet.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 1280  # EfficientNet-B0 final feature dim

        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False
            # Unfreeze last 2 blocks for fine-tuning
            for p in self.features[-2:].parameters():
                p.requires_grad = True

    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)  # (B, 1280)


class PretrainedShallowBranch(nn.Module):
    """MobileNetV3 early layers as shallow feature extractor (replaces CNN_Model2).
    Uses only the first 4 blocks → output: (B, 40) features."""
    def __init__(self, freeze_backbone=False):
        super().__init__()
        mobilenet = tv_models.mobilenet_v3_large(weights=tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        # Adapt first conv: 3→1 channel
        old = mobilenet.features[0][0]
        new = nn.Conv2d(1, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        with torch.no_grad():
            new.weight = nn.Parameter(old.weight.mean(dim=1, keepdim=True))
        mobilenet.features[0][0] = new
        # Take only first 4 blocks (shallow)
        self.features = mobilenet.features[:4]
        self.pool = nn.AdaptiveAvgPool2d(1)
        # Determine output dim
        with torch.no_grad():
            dummy = torch.randn(1, 1, 224, 224)
            self.out_dim = self.pool(self.features(dummy)).flatten(1).shape[1]
        print(f"  ShallowBranch out_dim = {self.out_dim}")

        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False

    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)  # (B, out_dim)


class HybridPretrainedCCAL(nn.Module):
    """Paper's MHA head with pretrained conv blocks.
    Deep branch: EfficientNet-B0 features (1280-dim)
    Shallow branch: MobileNetV3 early layers (40-dim)
    Head: project → MHA×2 → GAP → MLP → 3 classes"""
    def __init__(self, num_classes=NUM_CLASSES, num_heads=4, key_dim=16,
                 num_tokens=8, dropout_p=0.3, freeze_backbone=False):
        super().__init__()
        self.deep = PretrainedDeepBranch(freeze_backbone=freeze_backbone)
        self.shallow = PretrainedShallowBranch(freeze_backbone=freeze_backbone)
        concat_dim = self.deep.out_dim + self.shallow.out_dim

        embed_dim = num_heads * key_dim  # 64
        self.embed_dim  = embed_dim
        self.num_tokens = num_tokens

        # Project concatenated features → tokens
        self.feature_projection = nn.Linear(concat_dim, num_tokens * embed_dim)

        # Paper's MHA head (identical)
        self.mha1 = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                                          dropout=0.1, batch_first=True)
        self.mha2 = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                                          dropout=0.1, batch_first=True)

        # Paper's MLP head (identical)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.batch_norm1 = nn.BatchNorm1d(embed_dim)
        self.dense1      = nn.Linear(embed_dim, 256)
        self.gelu        = nn.GELU()
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.dropout     = nn.Dropout(dropout_p)
        self.ffn         = nn.Linear(256, num_classes)

        print(f"  HybridCCAL: concat_dim={concat_dim}, embed_dim={embed_dim}, tokens={num_tokens}")

    def forward(self, x):
        f_deep    = self.deep(x)                                       # (B, 1280)
        f_shallow = self.shallow(x)                                    # (B, 40)
        concat    = torch.cat([f_deep, f_shallow], dim=1)              # (B, 1320)

        proj   = self.feature_projection(concat)                       # (B, T*E)
        tokens = proj.view(proj.size(0), self.num_tokens, self.embed_dim)  # (B,T,E)

        attn1, _ = self.mha1(tokens, tokens, tokens)
        attn2, _ = self.mha2(attn1, attn1, attn1)

        out = attn2.transpose(1, 2)                                    # (B,E,T)
        out = self.global_avg_pool(out).squeeze(-1)                    # (B,E)

        out = self.batch_norm1(out)
        out = self.dense1(out)
        out = self.gelu(out)
        out = self.batch_norm2(out)
        out = self.dropout(out)
        return self.ffn(out)  # logits


# Sanity check
_h = HybridPretrainedCCAL()
_n = sum(p.numel() for p in _h.parameters())
_out = _h(torch.randn(2, 1, 224, 224))
print(f"\n✅ HybridPretrainedCCAL | params={_n:,} ({_n/1e6:.2f}M) | output={_out.shape}")
del _h, _n, _out

---
## 4 · Run Experiment 1C-i — EfficientNet-B0 (Standalone)

In [ ]:
print("=" * 68)
print(" EXPERIMENT 1C-i(a) — EfficientNet-B0 (Pretrained, Fine-tuned)")
print("=" * 68)

train_ld, val_ld, test_ld = make_loaders(split_records, augment_train=True)

effnet_metrics = run_experiment(
    model         = build_efficientnet_b0(),
    train_loader  = train_ld, val_loader = val_ld, test_loader = test_ld,
    save_dir      = RESULTS_DIR / '1ci_efficientnet_b0',
    model_name    = 'EfficientNet_B0',
    class_weights = class_weights,
)
print(f"\n📊 EfficientNet-B0 — Accuracy: {effnet_metrics['accuracy']*100:.2f}%  "
      f"F1: {effnet_metrics['f1_macro']:.4f}  "
      f"Benign Recall: {effnet_metrics['per_class']['Benign']['recall']:.4f}")

---
## 5 · Run Experiment 1C-i — MobileNetV3-Large (Standalone)

In [ ]:
print("=" * 68)
print(" EXPERIMENT 1C-i(b) — MobileNetV3-Large (Pretrained, Fine-tuned)")
print("=" * 68)

mobilenet_metrics = run_experiment(
    model         = build_mobilenet_v3(),
    train_loader  = train_ld, val_loader = val_ld, test_loader = test_ld,
    save_dir      = RESULTS_DIR / '1ci_mobilenet_v3',
    model_name    = 'MobileNetV3_Large',
    class_weights = class_weights,
)
print(f"\n📊 MobileNetV3-Large — Accuracy: {mobilenet_metrics['accuracy']*100:.2f}%  "
      f"F1: {mobilenet_metrics['f1_macro']:.4f}  "
      f"Benign Recall: {mobilenet_metrics['per_class']['Benign']['recall']:.4f}")

---
## 6 · Run Experiment 1C-ii — Hybrid: Pretrained Conv + Paper's MHA Head

**Key question:** Does the paper's attention head benefit from better features?

In [ ]:
print("=" * 68)
print(" EXPERIMENT 1C-ii — Hybrid: Pretrained Conv Blocks + Paper MHA Head")
print("   Deep branch: EfficientNet-B0 features (1280d)")
print("   Shallow branch: MobileNetV3 early layers (~40d)")
print("   Head: Paper's MHA×2 + MLP (identical)")
print("=" * 68)

hybrid_metrics = run_experiment(
    model         = HybridPretrainedCCAL(freeze_backbone=False),
    train_loader  = train_ld, val_loader = val_ld, test_loader = test_ld,
    save_dir      = RESULTS_DIR / '1cii_hybrid_pretrained_mha',
    model_name    = 'Hybrid_PretrainedCCAL',
    class_weights = class_weights,
    epochs        = 30,
)
print(f"\n📊 Hybrid — Accuracy: {hybrid_metrics['accuracy']*100:.2f}%  "
      f"F1: {hybrid_metrics['f1_macro']:.4f}  "
      f"Benign Recall: {hybrid_metrics['per_class']['Benign']['recall']:.4f}")

---
## 7 · Experiment 1C-iii — MHA Variants (TODO)

> **Placeholder** — will implement after baseline results are in.
>
> Ideas:
> - More heads (8 instead of 4)
> - Larger embedding dim (128 instead of 64)
> - Cross-attention between branches instead of self-attention on concatenated features
> - Add LayerNorm before MHA (pre-norm style)

In [ ]:
print("ℹ️  Experiment 1C-iii (MHA variants) is a TODO — implement after baselines are established.")

---
## 8 · Summary Comparison Table

In [ ]:
print("\n" + "=" * 80)
print(" EXPERIMENT 1C — ALL RESULTS SUMMARY")
print("=" * 80)
print(f"{'Model':<30} {'Params':>8} {'Accuracy':>10} {'F1-Macro':>10} {'Benign R':>10} {'AUC':>8}")
print("-" * 80)

all_results = [
    ('EfficientNet-B0',    effnet_metrics),
    ('MobileNetV3-Large',  mobilenet_metrics),
    ('Hybrid Pretrained+MHA', hybrid_metrics),
]

for name, m in all_results:
    print(f"{name:<30} {m.get('params_M','?'):>7}M "
          f"{m['accuracy']*100:>9.2f}% "
          f"{m['f1_macro']:>10.4f} "
          f"{m['per_class']['Benign']['recall']:>9.4f} "
          f"{m['auc_macro']:>8.4f}")

print("=" * 80)
print("\n─── To compare with 1A/1B: ───")
print("  Load metrics.json from ../exp_1a_paper_exact/ and ../exp_1b_paper_sequential/")
print("  to build the full thesis comparison table.")